# Actividad 8 — Línea base de SentimentAI Banking

Ejecuta todas las celdas en Colab. La prueba mide inferencia individual en el entorno del cuaderno. Se reutilizan los 300 ejemplos sintéticos de la Actividad 4, por lo que la calidad medida verifica la reproducción y sirve de referencia para los incidentes del ejercicio.

In [ ]:
%pip -q install mlflow transformers sentencepiece scikit-learn

In [ ]:
import json
import time
from pathlib import Path
from uuid import uuid4

import mlflow
import pandas as pd
import torch
from sklearn.metrics import f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DATA_URL = "https://raw.githubusercontent.com/riveranelizabeth-ship-it/Actividad-4/main/data/SentimentAI_Banking_Dataset_300.csv"
MODEL = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
REVISION = "f2f1202b1bdeb07342385c3f807f9c07cd8f5cf8"
LIMIT_MS = 500
SLO = 0.95

data = pd.read_csv(DATA_URL)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, revision=REVISION).to(device).eval()
labels = {"negative": "negativo", "neutral": "neutral", "positive": "positivo"}
print(f"Entorno: {device}. Comentarios: {len(data)}")

In [ ]:
def infer(comment):
    started = time.perf_counter()
    batch = tokenizer(comment, truncation=True, max_length=128, return_tensors="pt").to(device)
    if device.type == "cuda":
        torch.cuda.synchronize()
    encoded = time.perf_counter()
    with torch.inference_mode():
        index = int(model(**batch).logits.argmax(dim=-1).item())
    predicted = time.perf_counter()
    label = labels[model.config.id2label[index].lower()]
    return label, {
        "preprocesamiento_ms": (encoded - started) * 1000,
        "inferencia_ms": (predicted - encoded) * 1000,
    }


# Calentamiento fuera de la muestra de latencia.
for comment in data["comentario"].head(5):
    infer(comment)

records = []
for row in data.itertuples(index=False):
    trace_id = uuid4().hex
    timestamp = pd.Timestamp.now(tz="UTC").isoformat()
    started = time.perf_counter()
    try:
        prediction, spans = infer(row.comentario)
        status, error = "ok", None
    except Exception as exc:
        prediction, spans = "error", {}
        status, error = "error", type(exc).__name__
    records.append({
        "timestamp_utc": timestamp,
        "trace_id": trace_id,
        "modelo_revision": REVISION,
        "ejemplo_id": row.id,
        "etiqueta": row.sentimiento,
        "prediccion": prediction,
        "duracion_ms": (time.perf_counter() - started) * 1000,
        "estado": status,
        "tipo_error": error,
        **spans,
    })

results = pd.DataFrame(records)
negative = results[results["etiqueta"].eq("negativo")]
good = results["estado"].eq("ok") & results["duracion_ms"].le(LIMIT_MS)
bad_count = int((~good).sum())
allowed = round(1 - SLO, 6) * len(results)
metrics = {
    "solicitudes": len(results),
    "errores": int(results["estado"].eq("error").sum()),
    "latencia_p95_ms": float(results["duracion_ms"].quantile(0.95)),
    "fraccion_cumple_slo": float(good.mean()),
    "consumo_presupuesto": bad_count / allowed,
    "recall_negativo": float(negative["prediccion"].eq("negativo").mean()),
    "f1_macro": float(f1_score(results["etiqueta"], results["prediccion"],
                              labels=list(labels.values()), average="macro", zero_division=0)),
}
display(pd.Series(metrics, name="Línea base"))

In [ ]:
output = Path("evidencias_linea_base")
output.mkdir(exist_ok=True)
results.to_csv(output / "solicitudes.csv", index=False)
(output / "metricas.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

mlflow.set_tracking_uri(f"sqlite:///{Path('actividad8_mlflow.db').resolve()}")
mlflow.set_experiment("Actividad 8 - SentimentAI Banking")
with mlflow.start_run(run_name="linea-base") as run:
    mlflow.log_params({
        "modelo": MODEL, "revision": REVISION, "dispositivo": str(device),
        "umbral_ms": LIMIT_MS, "slo": SLO,
    })
    mlflow.log_metrics(metrics)
    mlflow.log_artifacts(str(output), artifact_path="linea_base")
    print(f"MLflow run ID: {run.info.run_id}")

print("Evidencias:", output.resolve())